In [ ]:
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.feature_selection import SelectFromModel, RFE

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import category_encoders as ce
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

In [2]:
df_loan_train = pd.read_csv('../data/processed_data/train_data_processed.csv')
df_loan_test = pd.read_csv('../data/processed_data/test_data_processed.csv')
df_loan_test.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,TARGET,AMT_REQ_CREDIT_BUREAU_HOUR_input,AMT_REQ_CREDIT_BUREAU_DAY_input,AMT_REQ_CREDIT_BUREAU_WEEK_input,AMT_REQ_CREDIT_BUREAU_MON_input,AMT_REQ_CREDIT_BUREAU_QRT_input,AMT_REQ_CREDIT_BUREAU_YEAR_input
0,159213,Cash loans,F,N,Y,0.0,67500.0,675000.0,19476.0,675000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,151735,Cash loans,M,Y,Y,2.0,315000.0,1288350.0,37800.0,1125000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
2,381923,Cash loans,F,N,N,1.0,112500.0,450000.0,25834.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,432341,Cash loans,M,Y,Y,0.0,270000.0,1079581.5,52065.0,886500.0,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,3.0
4,273574,Cash loans,F,N,N,0.0,90000.0,204858.0,14386.5,171000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0


In [3]:
lista_variables_codificar = df_loan_train.select_dtypes('object').columns.to_list()

X_train = df_loan_train.drop('TARGET',axis=1)
X_test = df_loan_test.drop('TARGET',axis=1)
y_train = df_loan_train['TARGET']
y_test = df_loan_test['TARGET']

# Definir las columnas para Label Encoding
label_encoding_vars = ['WEEKDAY_APPR_PROCESS_START']
target_encoding_vars = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
one_hot_encoding_vars = [var for var in lista_variables_codificar if var not in target_encoding_vars + label_encoding_vars]

In [4]:
from sklearn.preprocessing import LabelEncoder

# Crear el codificador
label_encoder = LabelEncoder()

# Aplicar Label Encoding en WEEKDAY_APPR_PROCESS_START en el conjunto de entrenamiento
X_train['WEEKDAY_APPR_PROCESS_START'] = label_encoder.fit_transform(X_train['WEEKDAY_APPR_PROCESS_START'])

# Aplicar el mismo mapeo en el conjunto de prueba
X_test['WEEKDAY_APPR_PROCESS_START'] = label_encoder.transform(X_test['WEEKDAY_APPR_PROCESS_START'])

# Verificar el resultado
print("Valores únicos tras codificación en entrenamiento:", X_train['WEEKDAY_APPR_PROCESS_START'].unique())
print("Valores únicos tras codificación en prueba:", X_test['WEEKDAY_APPR_PROCESS_START'].unique())

Valores únicos tras codificación en entrenamiento: [1 0 5 4 6 2 3]
Valores únicos tras codificación en prueba: [0 3 1 4 5 6 2]


In [ ]:
# Crear una clase personalizada para manejar el Target Encoding dentro del pipeline
class TargetEncodingWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.encoder = ce.TargetEncoder(cols=self.cols)

    def fit(self, X, y):
        self.encoder.fit(X[self.cols], y)
        self.feature_names_out = self.cols  # Guardar los nombres de las columnas
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = self.encoder.transform(X[self.cols])
        return X

    def get_feature_names_out(self, input_features=None):
        # Devuelve los nombres de las columnas transformadas
        return np.array(self.feature_names_out)

# Crear el pipeline
pipeline = Pipeline([
    ('preprocessor', ColumnTransformer(
        transformers=[

            # Target Encoding
            ('target_encoder', TargetEncodingWrapper(target_encoding_vars), target_encoding_vars), 

            # One-Hot Encoding
            ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore'), one_hot_encoding_vars)
        ],
        remainder='passthrough'  # Mantener el resto de las columnas sin procesar
    )),

    # Escalado de todas las variables transformadas
    ('scaler', StandardScaler()), 

    # Selección de variables con SelectFromModel usando Ridge
    ('feature_selection', SelectFromModel(LogisticRegression(C=1, penalty='l2'), threshold=0.1))
])

# Ajustar el pipeline con los datos de entrenamiento (requiere y_train)
pipeline.fit(X_train, y_train)

# Transformar los datos de entrenamiento y prueba
X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Obtener los nombres de las columnas transformadas por el preprocesador
all_feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

# Limpiar los nombres para quedarte solo con lo que va después de '__'
cleaned_feature_names = [col.split("__")[1] if "__" in col else col for col in all_feature_names]

# Obtener las columnas seleccionadas por SelectFromModel
selected_features_mask = pipeline.named_steps['feature_selection'].get_support()
selected_feature_names = np.array(cleaned_feature_names)[selected_features_mask]

# Convertir a DataFrame con las columnas seleccionadas
X_train_transformed = pd.DataFrame(X_train_transformed, columns=selected_feature_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=selected_feature_names)

# Verificar las primeras filas
print(X_train_transformed.head())

/opt/miniconda3/envs/aa_p2_eda/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


   ORGANIZATION_TYPE  NAME_INCOME_TYPE_Commercial associate  \
0           1.085138                              -0.551515   
1          -0.787774                              -0.551515   
2           0.596250                              -0.551515   
3           0.170113                              -0.551515   
4          -1.342503                              -0.551515   

   NAME_INCOME_TYPE_Pensioner  NAME_INCOME_TYPE_State servant  \
0                   -0.468609                       -0.274801   
1                   -0.468609                        3.639003   
2                   -0.468609                       -0.274801   
3                   -0.468609                        3.639003   
4                    2.133976                       -0.274801   

   NAME_INCOME_TYPE_Working  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE  \
0                  0.967890   -0.374783    -0.696770        -0.605275   
1                 -1.033175    1.204501     0.324029         1.105495   
2          

In [7]:
X_test_transformed

,ORGANIZATION_TYPE,NAME_INCOME_TYPE_Commercial associate,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Working,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_EMPLOYED,FLAG_EMP_PHONE,...,ENTRANCES_AVG,NONLIVINGAREA_AVG,BASEMENTAREA_MODE,ELEVATORS_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,ENTRANCES_MEDI,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,FLAG_DOCUMENT_3
0,-1.342503,-0.551515,2.133976,-0.274801,-1.033175,0.190390,-0.526741,0.372308,2.133384,-2.133389,...,-1.147701e+00,0.421478,-2.537921e-12,0.465832,-0.350363,-4.377255e-12,-1.136847e+00,-0.612304,-0.610261,-1.562480
1,-0.075662,1.813187,-0.468609,-0.274801,-1.033175,1.717855,0.741317,1.594287,-0.476421,0.468738,...,3.195389e-01,-0.230883,1.423799e+00,1.698395,1.089858,1.340389e+00,3.256144e-01,1.115970,1.133141,0.640008
2,0.628508,-0.551515,-0.468609,-0.274801,0.967890,-0.369942,-0.086720,-0.238682,-0.473745,0.468738,...,1.788198e+00,-0.305438,6.753002e-01,-0.356557,0.210363,5.984821e-01,1.789490e+00,-0.612304,-0.610261,0.640008
3,-0.567626,-0.551515,-0.468609,-0.274801,0.967890,1.197946,1.728484,0.946638,-0.464395,0.468738,...,4.724098e+00,-0.195676,4.820549e+00,5.397104,5.366993,4.700682e+00,4.715826e+00,-0.612304,-0.610261,0.640008
4,-1.342503,-0.551515,2.133976,-0.274801,-1.033175,-0.980435,-0.878944,-0.996309,2.133384,-2.133389,...,-1.147701e+00,-0.230883,-2.537921e-12,-0.356557,-1.095437,-4.377255e-12,-1.136847e+00,-0.180235,-0.174411,-1.562480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61498,-1.342503,-0.551515,2.133976,-0.274801,-1.033175,-0.874241,-1.018144,-0.788572,2.133384,-2.133389,...,-1.637255e+00,-0.305438,-1.982550e-01,-0.356557,-0.375967,-2.642448e-01,-1.624805e+00,-0.612304,-0.610261,-1.562480
61499,0.635865,1.813187,-0.468609,-0.274801,-1.033175,0.414523,0.772458,0.616704,-0.482585,0.468738,...,-1.014952e-12,-0.230883,-2.537921e-12,-0.356557,-0.196740,-4.377255e-12,2.007981e-12,-0.612304,-0.610261,-1.562480
61500,1.085675,1.813187,-0.468609,-0.274801,-1.033175,2.150969,4.677528,2.205277,-0.475784,0.468738,...,-1.700142e-01,-0.305438,6.072548e-01,1.287200,0.356305,5.324089e-01,-1.623441e-01,-0.612304,-0.610261,0.640008
61501,-0.552996,1.813187,-0.468609,-0.274801,-1.033175,-0.856310,-0.935621,-0.849671,-0.464954,0.468738,...,4.813042e-01,0.021778,-3.692879e-01,0.465832,1.062974,-8.867895e-02,8.135728e-01,-0.612304,-0.610261,0.640008
